In [1]:
import os, shutil

INPUT_DIR = "/kaggle/input/datasets/henilsinhraj/infer-gu-onnx"   
BASE      = "/kaggle/working"
EXPORT_DIR = f"{BASE}/exports"
MODEL_DIR  = f"{BASE}/model"

os.makedirs(EXPORT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR,  exist_ok=True)


for f in ["fs2_encoder.onnx", "fs2_decoder.onnx", "hifigan.onnx"]:
    shutil.copy(f"{INPUT_DIR}/{f}", f"{EXPORT_DIR}/{f}")
    print(f"{f} → {EXPORT_DIR}/")


for f in ["config.yaml", "feats_stats.npz"]:
    shutil.copy(f"{INPUT_DIR}/{f}", f"{MODEL_DIR}/{f}")
    print(f"{f} → {MODEL_DIR}/")

print("\nll files are in /kaggle/working")

fs2_encoder.onnx → /kaggle/working/exports/
fs2_decoder.onnx → /kaggle/working/exports/
hifigan.onnx → /kaggle/working/exports/
config.yaml → /kaggle/working/model/
feats_stats.npz → /kaggle/working/model/

ll files are in /kaggle/working


In [2]:
import sys
from unittest.mock import MagicMock

sys.modules['num_to_words'] = MagicMock()
sys.modules['g2p_en']       = MagicMock()
sys.modules['NumberToText'] = MagicMock()

sys.modules.pop('text_preprocess_for_inference', None)

print("Mocked missing modules")

Mocked missing modules


In [3]:
!pip install onnxruntime --quiet

import os
BASE = "/kaggle/working"
RAW  = "https://github.com/smtiitm/Fastspeech2_HS/raw/main"

os.makedirs(f"{BASE}/phone_dict",   exist_ok=True)
os.makedirs(f"{BASE}/numToText",    exist_ok=True)

files = {
    f"{BASE}/text_preprocess_for_inference.py" : f"{RAW}/text_preprocess_for_inference.py",
    f"{BASE}/multilingualcharmap.json"          : f"{RAW}/multilingualcharmap.json",
    f"{BASE}/NumberToText.py"                   : f"{RAW}/NumberToText.py",
    f"{BASE}/numToText/gujarati.csv"            : f"{RAW}/numToText/gujarati.csv",
    f"{BASE}/phone_dict/gujarati"               : f"{RAW}/phone_dict/gujarati",
}

for dest, url in files.items():
    fname = os.path.basename(dest)
    os.system(f'wget -q "{url}" -O "{dest}"')
    size = os.path.getsize(dest) if os.path.exists(dest) else 0
    print(f"{fname}: {size/1024:.1f} KB")

import numpy as np, sys, os, yaml
import onnxruntime as ort
from scipy.io.wavfile import write as wav_write
from IPython.display import Audio, display

BASE        = "/kaggle/working"
INPUT_DIR   = "/kaggle/input/datasets/henilsinhraj/infer-gu-onnx"
SAMPLING_RATE = 22050
MAX_WAV_VALUE = 32768.0
sys.path.insert(0, BASE)

from text_preprocess_for_inference import TTSDurAlignPreprocessor
sess_enc  = ort.InferenceSession(f"{INPUT_DIR}/fs2_encoder.onnx",  providers=["CPUExecutionProvider"])
sess_dec  = ort.InferenceSession(f"{INPUT_DIR}/fs2_decoder.onnx",  providers=["CPUExecutionProvider"])
sess_hifi = ort.InferenceSession(f"{INPUT_DIR}/hifigan.onnx",      providers=["CPUExecutionProvider"])
print(" All 3 ONNX models loaded")

with open(f"{INPUT_DIR}/config.yaml") as f:
    config = yaml.safe_load(f)
token_list = config["token_list"]
token2id   = {tok: idx for idx, tok in enumerate(token_list)}
print(f"Vocab — {len(token_list)} tokens")

stats    = np.load(f"{INPUT_DIR}/feats_stats.npz")
count    = stats["count"]
mel_mean = (stats["sum"] / count).astype(np.float32)
mel_std  = (np.sqrt(stats["sum_square"] / count - mel_mean ** 2)).astype(np.float32)
print(" Mel stats loaded")

preprocessor = TTSDurAlignPreprocessor()
print("Ready — fully ONNX, zero PyTorch\n")

def synthesize(text, label="out"):
    print(f"{text[:60]}")
    preprocessed_list, _ = preprocessor.preprocess(text, "gujarati", "male")
    preprocessed_text    = " ".join(preprocessed_list)

 
    token_ids = np.array(
        [[token2id.get(c, 1) for c in preprocessed_text if c != ' ']],
        dtype=np.int64
    )


    hs, d_log = sess_enc.run(None, {"text_ids": token_ids})

    durations = np.clip(np.round(np.exp(d_log[0])), 0, None).astype(int)
    print(f"  Frames: {durations.sum()}")


    hs_exp = np.concatenate(
        [np.repeat(hs[0][i:i+1], durations[i], axis=0) for i in range(hs.shape[1])],
        axis=0
    )[np.newaxis, :].astype(np.float32)


    mel_norm   = sess_dec.run(None, {"hidden_states": hs_exp})[0]
    mel_denorm = (mel_norm * mel_std[None,:,None] + mel_mean[None,:,None]).astype(np.float32)
    mel_scaled = (mel_denorm * 2.3262).astype(np.float32)

    wav   = sess_hifi.run(None, {"mel_spectrogram": mel_scaled})[0].squeeze()
    wav   = wav / (np.abs(wav).max() + 1e-8)
    audio = (wav * MAX_WAV_VALUE).astype(np.int16)

    os.makedirs(f"{BASE}/output", exist_ok=True)
    path = f"{BASE}/output/{label}.wav"
    wav_write(path, SAMPLING_RATE, audio)
    print(f"{len(audio)/SAMPLING_RATE:.2f}s → {path}")
    display(Audio(path, rate=SAMPLING_RATE))

synthesize("નમસ્તે, તમે કેમ છો?",                                          label="short")
synthesize("ભારત એક મહાન દેશ છે અને અહીં અનેક ભાષાઓ બોલવામાં આવે છે.", label="medium")
synthesize("આજે હવામાન ખૂબ સારું છે. બાળકો બગીચામાં રમી રહ્યા છે.",     label="long")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 80.3 MB/s eta 0:00:00:00:0100:01
text_preprocess_for_inference.py: 34.8 KB
multilingualcharmap.json: 23.4 KB
NumberToText.py: 3.4 KB
gujarati.csv: 2.2 KB
gujarati: 3509.2 KB


/kaggle/working/text_preprocess_for_inference.py:68: SyntaxWarning: invalid escape sequence '\.'
  """[?;:)(!|&’‘,।\."]""": "",


Phone dictionary loaded for the following languages: ['gujarati']
Loading G2P model... Done!
Phone dictionary loaded for the following languages: ['gujarati']
Loading G2P model... Done!
Phone dictionary loaded for the following languages: ['gujarati']
Loading G2P model... Done!
Phone dictionary loaded for the following languages: ['gujarati']
Loading G2P model... Done!
Phone dictionary loaded for the following languages: ['gujarati']
Loading G2P model... Done!
 All 3 ONNX models loaded
Vocab — 56 tokens
 Mel stats loaded
Ready — fully ONNX, zero PyTorch

નમસ્તે, તમે કેમ છો?
નમસ્તે, તમે કેમ છો?
cleaned text નમસ્તે#  તમે કેમ છો
word not in dict: []
['$namastE.', '$tamEkEmCo.']
  Frames: 265
3.08s → /kaggle/working/output/short.wav


ભારત એક મહાન દેશ છે અને અહીં અનેક ભાષાઓ બોલવામાં આવે છે.
ભારત એક મહાન દેશ છે અને અહીં અનેક ભાષાઓ બોલવામાં આવે છે.
cleaned text ભારત એક મહાન દેશ છે અને અહીં અનેક ભાષાઓ બોલવામાં આવે છે# 
word not in dict: []
['$BAratEkmahAndEशCEanEahIqanEkBAषAobolwAmAqAwECE.']
  Frames: 605
7.02s → /kaggle/working/output/medium.wav


આજે હવામાન ખૂબ સારું છે. બાળકો બગીચામાં રમી રહ્યા છે.
આજે હવામાન ખૂબ સારું છે. બાળકો બગીચામાં રમી રહ્યા છે.
cleaned text આજે હવામાન ખૂબ સારું છે#  બાળકો બગીચામાં રમી રહ્યા છે# 
word not in dict: []
['$AjEhawAmAnखUbsAruqCE.', '$bAളkobagIcAmAqramIrahyACE.']
  Frames: 669
7.77s → /kaggle/working/output/long.wav


In [4]:
from onnxruntime.quantization import quantize_dynamic, QuantType
import os

BASE       = "/kaggle/working"
EXPORT_DIR = f"{BASE}/exports"

def show_size(label, path):
    mb = os.path.getsize(path) / 1e6
    print(f"  {label}: {mb:.1f} MB")

print("Original sizes:")
show_size("fs2_encoder", f"{EXPORT_DIR}/fs2_encoder.onnx")
show_size("fs2_decoder", f"{EXPORT_DIR}/fs2_decoder.onnx")
show_size("hifigan",     f"{EXPORT_DIR}/hifigan.onnx")

print("\nQuantizing...")

import shutil
shutil.copy(f"{EXPORT_DIR}/fs2_encoder.onnx", f"{EXPORT_DIR}/fs2_encoder_q.onnx")
print("Encoder — kept float32")
# Int 8 decoder
quantize_dynamic(
    model_input  = f"{EXPORT_DIR}/fs2_decoder.onnx",
    model_output = f"{EXPORT_DIR}/fs2_decoder_q.onnx",
    weight_type  = QuantType.QInt8,
)
print("Decoder quantized")

# HiFi-GAN — INT8
quantize_dynamic(
    model_input  = f"{EXPORT_DIR}/hifigan.onnx",
    model_output = f"{EXPORT_DIR}/hifigan_q.onnx",
    weight_type  = QuantType.QInt8,
)
print("HiFi-GAN quantized")

print("\nQuantized sizes:")
show_size("fs2_encoder_q", f"{EXPORT_DIR}/fs2_encoder_q.onnx")
show_size("fs2_decoder_q", f"{EXPORT_DIR}/fs2_decoder_q.onnx")
show_size("hifigan_q",     f"{EXPORT_DIR}/hifigan_q.onnx")

orig  = sum(os.path.getsize(f"{EXPORT_DIR}/{f}.onnx") for f in ["fs2_encoder","fs2_decoder","hifigan"])
quant = sum(os.path.getsize(f"{EXPORT_DIR}/{f}.onnx") for f in ["fs2_encoder_q","fs2_decoder_q","hifigan_q"])
print(f"\nTotal original  : {orig/1e6:.1f} MB")
print(f"Total quantized : {quant/1e6:.1f} MB")
print(f"Reduction       : {(1-quant/orig)*100:.1f}%")

Original sizes:
  fs2_encoder: 85.2 MB
  fs2_decoder: 78.8 MB
  hifigan: 55.7 MB

Quantizing...
Encoder — kept float32


Decoder quantized
HiFi-GAN quantized

Quantized sizes:
  fs2_encoder_q: 85.2 MB
  fs2_decoder_q: 32.8 MB
  hifigan_q: 22.1 MB

Total original  : 219.7 MB
Total quantized : 140.0 MB
Reduction       : 36.3%


# Trying onnx quantized model

In [6]:
import numpy as np, sys, os, yaml
import onnxruntime as ort
from scipy.io.wavfile import write as wav_write
from IPython.display import Audio, display

BASE        = "/kaggle/working"
EXPORT_DIR  = f"{BASE}/exports"
MODEL_DIR   = "/kaggle/working/model/"
SAMPLING_RATE = 22050
MAX_WAV_VALUE = 32768.0
sys.path.insert(0, BASE)

from text_preprocess_for_inference import TTSDurAlignPreprocessor


sess_enc  = ort.InferenceSession(f"{EXPORT_DIR}/fs2_encoder_q.onnx",  providers=["CPUExecutionProvider"])
sess_dec  = ort.InferenceSession(f"{EXPORT_DIR}/fs2_decoder_q.onnx",  providers=["CPUExecutionProvider"])
sess_hifi = ort.InferenceSession(f"{EXPORT_DIR}/hifigan_q.onnx",      providers=["CPUExecutionProvider"])
print("Quantized models loaded")

with open(f"{MODEL_DIR}/config.yaml") as f:
    config = yaml.safe_load(f)
token_list = config["token_list"]
token2id   = {tok: idx for idx, tok in enumerate(token_list)}

stats    = np.load(f"{MODEL_DIR}/feats_stats.npz")
count    = stats["count"]
mel_mean = (stats["sum"] / count).astype(np.float32)
mel_std  = (np.sqrt(stats["sum_square"] / count - mel_mean ** 2)).astype(np.float32)

preprocessor = TTSDurAlignPreprocessor()
print("Ready\n")

def synthesize(text, label="test"):
    print(f"[{label}] {text[:60]}")
    preprocessed_list, _ = preprocessor.preprocess(text, "gujarati", "male")
    preprocessed_text    = " ".join(preprocessed_list)

    token_ids = np.array(
        [[token2id.get(c, 1) for c in preprocessed_text if c != ' ']],
        dtype=np.int64
    )

    hs, d_log   = sess_enc.run(None, {"text_ids": token_ids})
    durations   = np.clip(np.round(np.exp(d_log[0])), 0, None).astype(int)
    print(f"  Frames: {durations.sum()}")

    hs_exp = np.concatenate(
        [np.repeat(hs[0][i:i+1], durations[i], axis=0) for i in range(hs.shape[1])],
        axis=0
    )[np.newaxis, :].astype(np.float32)

    mel_norm   = sess_dec.run(None, {"hidden_states": hs_exp})[0]
    mel_denorm = (mel_norm * mel_std[None,:,None] + mel_mean[None,:,None]).astype(np.float32)
    mel_scaled = (mel_denorm * 2.3262).astype(np.float32)

    wav   = sess_hifi.run(None, {"mel_spectrogram": mel_scaled})[0].squeeze()
    wav   = wav / (np.abs(wav).max() + 1e-8)
    audio = (wav * MAX_WAV_VALUE).astype(np.int16)

    path = f"{BASE}/output/quantized_{label}.wav"
    os.makedirs(f"{BASE}/output", exist_ok=True)
    wav_write(path, SAMPLING_RATE, audio)
    print(f"{len(audio)/SAMPLING_RATE:.2f}s → {path}")
    display(Audio(path, rate=SAMPLING_RATE))

synthesize("નમસ્તે, તમે કેમ છો?",                                          label="short")
synthesize("ભારત એક મહાન દેશ છે અને અહીં અનેક ભાષાઓ બોલવામાં આવે છે.", label="medium")
synthesize("આજે હવામાન ખૂબ સારું છે. બાળકો બગીચામાં રમી રહ્યા છે.",     label="long")

Quantized models loaded
Ready

[short] નમસ્તે, તમે કેમ છો?
નમસ્તે, તમે કેમ છો?
cleaned text નમસ્તે#  તમે કેમ છો
word not in dict: []
['$namastE.', '$tamEkEmCo.']
  Frames: 265
3.08s → /kaggle/working/output/quantized_short.wav


[medium] ભારત એક મહાન દેશ છે અને અહીં અનેક ભાષાઓ બોલવામાં આવે છે.
ભારત એક મહાન દેશ છે અને અહીં અનેક ભાષાઓ બોલવામાં આવે છે.
cleaned text ભારત એક મહાન દેશ છે અને અહીં અનેક ભાષાઓ બોલવામાં આવે છે# 
word not in dict: []
['$BAratEkmahAndEशCEanEahIqanEkBAषAobolwAmAqAwECE.']
  Frames: 605
7.02s → /kaggle/working/output/quantized_medium.wav


[long] આજે હવામાન ખૂબ સારું છે. બાળકો બગીચામાં રમી રહ્યા છે.
આજે હવામાન ખૂબ સારું છે. બાળકો બગીચામાં રમી રહ્યા છે.
cleaned text આજે હવામાન ખૂબ સારું છે#  બાળકો બગીચામાં રમી રહ્યા છે# 
word not in dict: []
['$AjEhawAmAnखUbsAruqCE.', '$bAളkobagIcAmAqramIrahyACE.']
  Frames: 669
7.77s → /kaggle/working/output/quantized_long.wav
